# DSCI-598 Capstone
## Maryville University
### November - December 2023
### Alison Hawke

# Data Visualisations and analysis  

In [ ]:
import numpy as np 
import pandas as pd 
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

In [ ]:
train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv', dtype = str)
test = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv', dtype = str)

In [ ]:
print('Training set shape', train.shape)
print('Test set shape', test.shape)

## Feature data fields in the provided Kaggle dataset:

* Elevation in meters
* Aspect in degrees azimuth
* Slope in degrees
* Horizontal_Distance_To_Hydrology dist to nearest surface water features
* Vertical_Distance_To_Hydrology dist to nearest surface water features
* Horizontal_Distance_To_Roadways dist to nearest roadway
* Hillshade_9am (0 to 255 index) hillshade index at 9am, summer solstice
* Hillshade_Noon (0 to 255 index) hillshade index at noon, summer solstice
* Hillshade_3pm (0 to 255 index) hillshade index at 3pm, summer solstice
* Horizontal_Distance_To_Fire_Points dist to nearest wildfire ignition points
* Wilderness_Area (4 binary columns, 0 = absent or 1 = present) wilderness area designation
* Soil_Type (40 binary columns, 0 = absent or 1 = present) soil type designation

These features are used to predict the following label:
* Cover_Type (7 types) Forest cover type designation

The seven forest cover types are:

* Spruce/Fir
* Lodgepole Pine
* Ponderosa Pine
* Cottonwood/Willow
* Aspen
* Douglas-fir
* Krummholz

The features Soil_Type and Wilderness_Area have been one-hot encoded in the provided data. 

In [ ]:
train.head()

In [ ]:
df = train.drop(['Id'], axis = 1)
df.value_counts(['Cover_Type']).sort_index()

All cover types are equally represented in the training data set, which implies this set was curated to be useful for data science projects and may not be an accurate representation of reality.

In [ ]:
df.isnull().sum()

There are no null values in the data, another anomoly pointing to a highly curated training set.

In [ ]:
df.shape

# Numeric features 

In [ ]:
# extract numeric features from training set
num_features = train.iloc[:,1:11]

In [ ]:
num_features.describe()

# Categorical features

In [ ]:
# extract binary features from training set
cat_features = train.iloc[:, 11:-1]

In [ ]:
cat_features.describe()

# Correlation between features

The correlation matrix displays which features are strongly correlated to each other, displaying a pairwise list.

From the output below, Hillshade_3pm and Hillshade_9am are strongly correlated, it could be useful to exclude one of these measurements as they are similar and therefore do not add much to the data. Hillshade also correlates with Slope and Aspect, which is logical.

Horizontal distance to roadways and firepoints (wildfire ignition points) are correlated, this could be because roads and wildfires are linked due to careless human activity.

Correlation scores above 0.6 are considered a good correlation, which are the following:

* Wilderness_Area4 and Elevation: 0.783651
* Hillshade_3pm and Hillshade_9am: 0.779965
* Vertical_Distance_To_Hydrology and Horizontal_Distance_To_Hydrology: 0.652142
* Aspect and Hillshade_3pm: 0.635022
* Hillshade_3pm and Hillshade_Noon: 0.614526
* Hillshade_Noon and Slope: 0.612613

None of the soil types are included in this list.

This analysis was useful for the [feature engineering notebook](https://www.kaggle.com/alisonhawke/dsci-598-feature-changes) when choosing which features to combine, or bin.

In [ ]:
corrMatrix = df.corr()

corrAbs = corrMatrix.abs().unstack()
corrSorted = corrAbs.sort_values(ascending = False).drop_duplicates()

corrSorted.head(50)

## Correlation plot

In [ ]:
def show_corr_plot(df):
    corr = df[['Elevation', 'Aspect', 'Slope',
           'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology',
           'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon',
           'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points']].corr()

    up = corr.where(np.tril(np.ones(corr.shape), k = -1).astype(bool))
    sns.heatmap(up, fmt = '.1f', annot = True, cmap = 'plasma')
    
show_corr_plot(df)

# Attempt to reverse the one-hot encoding

While this was not sucessful in Kaggle, the idxmax method worked on a local notebook.

In [ ]:
wilderness_temp = train[['Wilderness_Area1', 'Wilderness_Area2', 'Wilderness_Area3', 'Wilderness_Area4']]
wilderness_temp.describe()

In [ ]:
#Use one of the following:
#wilderness_df = pd.from_dummies(wilderness_temp).rename(columns = {'': 'Wilderness_Area'})
#wilderness_df = temp.idxmax(axis = 1)

In [ ]:
soil_temp = train[['Soil_Type1', 'Soil_Type2', 'Soil_Type3', 'Soil_Type4', 
                      'Soil_Type5', 'Soil_Type6', 'Soil_Type7', 'Soil_Type8', 'Soil_Type9', 
                      'Soil_Type10', 'Soil_Type11', 'Soil_Type12', 'Soil_Type13', 'Soil_Type14', 
                      'Soil_Type15', 'Soil_Type16', 'Soil_Type17', 'Soil_Type18', 'Soil_Type19', 
                      'Soil_Type20', 'Soil_Type21', 'Soil_Type22', 'Soil_Type23', 'Soil_Type24',
                      'Soil_Type25', 'Soil_Type26', 'Soil_Type27', 'Soil_Type28', 'Soil_Type29',
                      'Soil_Type30', 'Soil_Type31', 'Soil_Type32', 'Soil_Type33', 'Soil_Type34',
                      'Soil_Type35', 'Soil_Type36', 'Soil_Type37', 'Soil_Type38', 'Soil_Type39',
                      'Soil_Type40']]
soil_temp.describe()

In [ ]:
#Use one of the following:
#soil_df = pd.from_dummies(soil_temp).rename(columns = {'': 'Soil_Type'})
#soil_df = soil_temp.idxmax(axis = 1)

# Wilderness Areas

The Wilderness Areas are well-distributed in the test set:

* Area 1: 3597 records (0.2379 of the data set)
* Area 2: 5174 records (0.3422 of the data set)
* Area 3: 6439 records (0.4259 of the data set)
* Area 4: 4675 records (0.3092 of the data set)

Areas 4, 1, and 3 are in the top 20 most important features list, in descending order of importance.

# Soil Types

* Soil types 7 and 15 do not appear in the training data set. 
* Types 8 and 25 only occur once. 
* Types 9, 21, 27, 28, and 36 occur less than 20 times.

Types 10, 38, 3, 39, 4, 40, and 30 are in the top 20 most important features list, in descending order of importance.

# Scatter plots

In [ ]:
train = pd.read_csv('../input/forest-cover-type-prediction/train.csv')
train = train.iloc[:,1:]

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(24,18))

sns.scatterplot(data=train, x="Horizontal_Distance_To_Hydrology", y="Vertical_Distance_To_Hydrology", hue="Cover_Type", legend='full', palette='rainbow_r', ax=ax[0,0])
sns.scatterplot(data=train, x="Elevation", y="Horizontal_Distance_To_Roadways", hue="Cover_Type", legend='full', palette='rainbow_r', ax=ax[0,1])
sns.scatterplot(data=train, x="Aspect", y="Hillshade_3pm", hue="Cover_Type", legend='full', palette='rainbow_r', ax=ax[1,0])
sns.scatterplot(data=train, x="Hillshade_3pm", y="Hillshade_Noon", hue="Cover_Type", legend='full', palette='rainbow_r', ax=ax[1,1])
plt.show()

**Top left scatter plot**

The plot shows vertical and horizontal distance to hydrology, with colours for each type of cover. The clustering in the corner of the graph shows a preference for most trees to be closer to water. One tree, the Krummholtz, seems to be comfortable regardless of water access.

**Top right scatter plot**

The plot shows elevation and horizontal distance to roadways, with colours for each type of cover. The banding in the plot shows the trees prefer certain elevations and all can grow close to the roads.

**Bottom left scatter plot**

The plot shows aspect and hill shade at 3pm, with colours for each type of cover. At an aspect of 200 degrees, there is a switch in the hill shade at 3pm, with a zenith of 100 and a peak of 250.

**Bottom right scatter plot**

The plot shows hill shade at 3pm and hill shade at noon, with colours for each type of cover. There is some clustering of cover types on the far left and right of the graph.

# Histograms

In [ ]:
train.iloc[:,:10].hist(figsize = (16, 12), bins = 50)
plt.show()

The histograms for elevation shows 3 distinct peaks.

The following histograms are skewed to the left:

* Aspect
* Slope
* Horizontal_Distance_To_Hydrology 
* Vertical_Distance_To_Hydrology
* Horizontal_Distance_To_Roadways
* Horizontal_Distance_To_Fire_Point


The following histograms are skewed to the right:

* Hillshade_9am 
* Hillshade_Noon
* Hillshade_3pm 


# Bar charts

In [ ]:
cat_features = train.iloc[:, 10:-1]
wild_data, soil_data = cat_features.iloc[:,:4], cat_features.iloc[:,4:]

sns.set_style("darkgrid", {'grid.color':'.1'})
wild_data.sum().plot(kind = 'bar', figsize = (10,8))
plt.title('# of Observations of Wilderness Areas', size = 16)
plt.xlabel('Wilderness Areas', size = 14)
plt.ylabel('# of Observations', size = 14)
plt.xticks(rotation = 'horizontal', size = 10)
plt.yticks(size = 12)

sns.despine()
plt.show()

Therew are the most data points for Wilderness Area 3, then area 4 and 1. Wilderness area 2 is not well-represented in the training data. 

In [ ]:
sns.set_style("darkgrid", {'grid_color': '.1'})

# sum soil data, pass it as a series
soil_sum = pd.Series(soil_data.sum())
soil_sum.sort_values(ascending = False, inplace = True)

# plot bar
soil_sum.plot(kind = 'barh', figsize = (20,18))
plt.gca().invert_yaxis()
plt.title('# of Observations of Soil Types', size = 18)
plt.xlabel('# of Observation', size = 16)
plt.ylabel('Soil Types', size = 16)
plt.xticks(rotation = 'horizontal',size = 14)
plt.yticks(size = 14)

sns.despine()
plt.show()

Soil types 8, 25, 7, snd 15, are barely present or not part of the training data. This could skew the outcomes of the models if those areas are key to determining a particular cover type.

Soil types 10, 29, and 3 are the most well-represented. 

# Feature exploration using a random forest model

There are no null values in this data set, making it a very well curated data set. 

From the list of features by importance, the physical location and access to water are the best predictors of cover type.

Submission score: 0.73625

In [ ]:
X_test = train.drop(['Cover_Type'], axis = 1)
y_test = train['Cover_Type']
test_df = test.drop(['Id'], axis = 1)

In [ ]:
print('Training set shape', X_test.shape)
print('Test set shape', y_test.shape)

In [ ]:
rf = RandomForestClassifier(n_estimators = 1000, bootstrap = True, oob_score = True) 
rf.fit(X_test, y_test)

In [ ]:
print('Score: ', rf.score(X_test, y_test))

In [ ]:
%%time

# Run Predictions on test dataset
pred = rf.predict(test_df.values) 

# Create submissions csv file 
Id = np.asarray(test['Id'])

submission = pd.DataFrame({'Id':Id, 'Cover_Type':pred})
submission.to_csv('submission.csv', header = True, index = False)

# Feature importance based on mean decrease in impurity

From the scikit-learn documentation on [feature importance](https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html):

"Feature importances are provided by the fitted attribute feature_importances_ and they are computed as the mean and standard deviation of accumulation of the impurity decrease within each tree."

In [ ]:
# code from:
# https://shankarmsy.github.io/posts/forest-cover-types.html
# modified for top 20 features

def importances(estimator, col_array, title):
# Calculate the feature ranking top 20 
    importances = estimator.feature_importances_ 
    indices = np.argsort(importances)[::-1] 

    print("%s Top 20 Important Features\n" %title)
    for f in range(20): 
        print("%d. %s (%f)" % (f + 1, col_array.columns[indices[f]], importances[indices[f]])) 

    # Mean Feature Importance 
    print("\nMean Feature Importance %.6f" %np.mean(importances))

    # Plot the feature importances of the forest 
    indices = indices[:20] 
    plt.figure() 
    plt.title(title+" Top 20 Feature importances") 
    plt.bar(range(20), importances[indices], align = "center") 
    plt.xticks(range(20), col_array.columns[indices], fontsize = 14, rotation = 90) 
    plt.xlim([-1, 20]) 
    plt.show() 

In [ ]:
importances(rf, df, "Cover Type (random forest model)") 

# Feature importance based on feature permutation

From the scikit-learn documentation on [feature importance](https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html):

"Permutation feature importance overcomes limitations of the impurity-based feature importance: they do not have a bias toward high-cardinality features and can be computed on a left-out test set."

This method does significantly increase the time required to determine feature importances.

In [ ]:
feature_names = list(X_test.columns)

start_time = time.time()
result = permutation_importance(rf, X_test, y_test, 
                                n_repeats = 10, random_state = 42, n_jobs = 2)
elapsed_time = time.time() - start_time

print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")

forest_importances = pd.Series(result.importances_mean, index = feature_names)

In [ ]:
indices = np.argsort(forest_importances)[::-1] 

print("Top 20 Important Features using feature permutation\n")
for f in range(20): 
    print("%d. %s (%f)" % (f + 1, df.columns[indices[f]], forest_importances[indices[f]])) 

# Plot the feature importances of the forest 
indices = indices[:20] 
plt.figure() 
plt.title("Top 20 Feature importances using feature permutation") 
plt.bar(range(20), forest_importances[indices], align = "center") 
plt.xticks(range(20), df.columns[indices], fontsize = 14, rotation = 90) 
plt.xlim([-1, 20]) 
plt.show() 

Comparing these two methods of determining feature importance shows there are some differences in the two lists, but the top 4 features are the same in each list: elevation, horizontal distance to roadways, fire points, and hydrology.

Modelling based only on these features would provide an approximation of cover types that could be both swift and accurate enough for many purposes.